# HW04 作业：正则表达式与结构化网页数据

- 学号：20244081108
- 姓名：王亨泰
- 运行环境：conda 环境 `pyda`，Python 3.11.17
- 依赖：re、requests、beautifulsoup4（bs4）
- 作业内容：第一部分 12 道正则表达式编程题；第二部分 从维基百科城市页面抓取气候表

In [1]:
# 身份信息（已按题目要求将模板中的学号替换为本人学号）
STUDENT_ID = 20244081108
STUDENT_NAME = "王亨泰"
print("学号：", STUDENT_ID)
print("姓名：", STUDENT_NAME)

学号： 20244081108
姓名： 王亨泰


## 第一部分：正则表达式

下面 12 道题各封装为一个函数，全部使用标准库 `re` 实现，并在每组后给出测试用例与断言。

In [2]:
import re

### 第 1–4 题：基础匹配（整串匹配 `fullmatch`）

In [3]:
# 1. 判断字符串是否仅包含英文字母和数字（至少一个字符）
def is_alphanumeric(s):
    return bool(re.fullmatch(r"[A-Za-z0-9]+", s))

# 2. 判断是否为「一个 a 后面跟着零个或多个 b」
def match_a_b_star(s):
    return bool(re.fullmatch(r"ab*", s))

# 3. 判断是否为「一个 a 后面跟着一个或多个 b」
def match_a_b_plus(s):
    return bool(re.fullmatch(r"ab+", s))

# 4. 判断是否以 a 开头、以 b 结尾，中间为任意字符
def match_a_start_b_end(s):
    return bool(re.fullmatch(r"a.*b", s))

In [4]:
# 第 1–4 题测试
print("1 仅含字母数字:")
for t in ["abc123", "Python3", "abc 123", "a-b", ""]:
    print("  ", repr(t), "->", is_alphanumeric(t))
assert is_alphanumeric("abc123") and not is_alphanumeric("abc 123") and not is_alphanumeric("")

print("2 a 后零或多个 b:")
for t in ["a", "ab", "abbb", "b", "ac"]:
    print("  ", repr(t), "->", match_a_b_star(t))
assert all(match_a_b_star(x) for x in ["a", "ab", "abbb"])
assert not match_a_b_star("b") and not match_a_b_star("ac")

print("3 a 后一个或多个 b:")
for t in ["a", "ab", "abbb", "b"]:
    print("  ", repr(t), "->", match_a_b_plus(t))
assert not match_a_b_plus("a") and match_a_b_plus("ab") and not match_a_b_plus("b")

print("4 a 开头 b 结尾:")
for t in ["ab", "axb", "aXYZb", "a", "axbxc"]:
    print("  ", repr(t), "->", match_a_start_b_end(t))
assert all(match_a_start_b_end(x) for x in ["ab", "axb", "aXYZb"])
assert not match_a_start_b_end("a") and not match_a_start_b_end("axbxc")
print("第 1–4 题断言全部通过")

1 仅含字母数字:
   'abc123' -> True
   'Python3' -> True
   'abc 123' -> False
   'a-b' -> False
   '' -> False
2 a 后零或多个 b:
   'a' -> True
   'ab' -> True
   'abbb' -> True
   'b' -> False
   'ac' -> False
3 a 后一个或多个 b:
   'a' -> False
   'ab' -> True
   'abbb' -> True
   'b' -> False
4 a 开头 b 结尾:
   'ab' -> True
   'axb' -> True
   'aXYZb' -> True
   'a' -> False
   'axbxc' -> False
第 1–4 题断言全部通过


### 第 5–8 题：替换与查找

In [5]:
# 5. 删除 IP 地址每一段前面的前导零（全零段保留一个 0）
def strip_ip_leading_zeros(ip):
    def clean(seg):
        return re.sub(r"^0+(?=\d)", "", seg)  # 仅当后面还有数字时才去掉前导零
    return ".".join(clean(seg) for seg in ip.split("."))

# 6. 空格与下划线互换（单遍处理，避免二次替换互相覆盖）
def swap_space_underscore(s):
    return re.sub(r"[ _]", lambda m: "_" if m.group(0) == " " else " ", s)

# 7. 日期格式 yyyy-mm-dd 转换为 dd-mm-yyyy
def convert_date(s):
    return re.sub(r"(\d{4})-(\d{2})-(\d{2})", r"\3-\2-\1", s)

# 8. 找出所有以字母 a 或 e 开头的单词（忽略大小写）
def words_starting_a_or_e(s):
    return re.findall(r"\b[ae]\w*", s, flags=re.IGNORECASE)

In [6]:
# 第 5–8 题测试
print("5 IP 去前导零:")
for ip in ["192.168.001.010", "10.0.000.00", "001.002.003.004"]:
    print("  ", ip, "->", strip_ip_leading_zeros(ip))
assert strip_ip_leading_zeros("192.168.001.010") == "192.168.1.10"
assert strip_ip_leading_zeros("10.0.000.00") == "10.0.0.0"
assert strip_ip_leading_zeros("001.002.003.004") == "1.2.3.4"

print("6 空格下划线互换:")
for t in ["hello world", "a_b c", "one two_three"]:
    print("  ", repr(t), "->", repr(swap_space_underscore(t)))
assert swap_space_underscore("hello world") == "hello_world"
assert swap_space_underscore("a_b c") == "a b_c"

print("7 日期转换:")
for t in ["2024-05-01", "1999-12-31"]:
    print("  ", t, "->", convert_date(t))
assert convert_date("2024-05-01") == "01-05-2024"
assert convert_date("1999-12-31") == "31-12-1999"

print("8 a/e 开头单词:")
sentence = "apple banana egg orange cherry Elephant ant"
print("  ", words_starting_a_or_e(sentence))
assert words_starting_a_or_e(sentence) == ["apple", "egg", "Elephant", "ant"]
print("第 5–8 题断言全部通过")

5 IP 去前导零:
   192.168.001.010 -> 192.168.1.10
   10.0.000.00 -> 10.0.0.0
   001.002.003.004 -> 1.2.3.4
6 空格下划线互换:
   'hello world' -> 'hello_world'
   'a_b c' -> 'a b_c'
   'one two_three' -> 'one_two three'
7 日期转换:
   2024-05-01 -> 01-05-2024
   1999-12-31 -> 31-12-1999
8 a/e 开头单词:
   ['apple', 'egg', 'Elephant', 'ant']
第 5–8 题断言全部通过


### 第 9–12 题：缩写、长度匹配与删除

In [7]:
# 9. 把独立单词 Road 缩写为 Rd（不影响 Roads 等词）
def abbreviate_road(s):
    return re.sub(r"\bRoad\b", "Rd", s)

# 10. 找出长度至少为 4 的单词
def words_at_least_four(s):
    return re.findall(r"\b\w{4,}\b", s)

# 11. 删除除字母和数字以外的所有字符
def remove_non_alnum(s):
    return re.sub(r"[^A-Za-z0-9]", "", s)

# 12. 删除括号及括号内的内容（含括号前多余空格）
def remove_parentheses(s):
    return re.sub(r"\s*\([^)]*\)", "", s)

In [8]:
# 第 9–12 题测试
print("9 Road 缩写:")
print("  ", abbreviate_road("Main Road and Road Avenue, Roads end"))
assert abbreviate_road("Main Road and Road Avenue, Roads end") == "Main Rd and Rd Avenue, Roads end"

print("10 至少 4 字符单词:")
print("  ", words_at_least_four("the quick brown fox jumps over"))
assert words_at_least_four("the quick brown fox jumps over") == ["quick", "brown", "jumps", "over"]

print("11 去非字母数字:")
for t in ["Hello, World!", "a@b#c 123"]:
    print("  ", repr(t), "->", repr(remove_non_alnum(t)))
assert remove_non_alnum("Hello, World!") == "HelloWorld"
assert remove_non_alnum("a@b#c 123") == "abc123"

print("12 去括号内容:")
lines = ["example (.com)", "w3resource", "github (.com)", "stackoverflow (.com)"]
for line in lines:
    print("  ", repr(line), "->", repr(remove_parentheses(line)))
assert [remove_parentheses(x) for x in lines] == ["example", "w3resource", "github", "stackoverflow"]
print("第 9–12 题断言全部通过")

9 Road 缩写:
   Main Rd and Rd Avenue, Roads end
10 至少 4 字符单词:
   ['quick', 'brown', 'jumps', 'over']
11 去非字母数字:
   'Hello, World!' -> 'HelloWorld'
   'a@b#c 123' -> 'abc123'
12 去括号内容:
   'example (.com)' -> 'example'
   'w3resource' -> 'w3resource'
   'github (.com)' -> 'github'
   'stackoverflow (.com)' -> 'stackoverflow'
第 9–12 题断言全部通过


## 第二部分：结构化网页数据（维基百科气候表）

任务要求：读取含有标题为 **"Climate data for ..."** 表格的维基百科城市页面，编写两个函数：

- `retrieve_climate_table(s)`：只接收一个 URL 字符串；页面含气候表时返回该表的 BeautifulSoup `Tag`；页面可访问但没有气候表时返回 `None`；发生网络或 HTTP 错误时抛出异常。
- `list_climate_table_row_names(s)`：只接收一个 URL，返回由各数据行**行名**组成的列表，每项为去除空白后的纯 Python 字符串（不含任何 Tag/HTML），并排除标题行、月份列标题行与来源行。

### 网络情况说明

当前网络环境下无法直接访问 en.wikipedia.org（连接被重置/超时）。为保证 notebook 可完整、可重复地运行，已将还原维基气候表真实结构的 HTML 快照随作业提交在 `fixtures/` 目录，并在本地启动一个临时 HTTP 服务，让上面两个函数**走完整的 URL 请求流程**进行验证；同时保留对真实维基 URL 的调用，以展示网络错误会被正常抛出。待网络可访问时，函数无需修改即可直接用于真实维基页面。

In [9]:
import requests
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; CourseworkBot/1.0; +https://example.edu/student)",
    "Accept-Language": "en-US,en;q=0.9",
}
MARKER = "Climate data for"


def _norm(text):
    # 去掉脚注引用标记 [1]，并把连续空白合并为单个空格
    text = re.sub(r"\s*\[\d+\]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _find_climate_table(html):
    """在已下载的 HTML 中查找气候表，返回 Tag 或 None。"""
    soup = BeautifulSoup(html, "html.parser")
    tables = soup.find_all("table")
    # 优先：<caption> 明确以 Climate data for 开头
    for table in tables:
        caption = table.find("caption")
        if caption and MARKER in _norm(caption.get_text(" ")):
            return table
    # 回退：整表文本包含 Climate data for（处理标题放在 <th colspan> 的页面）
    for table in tables:
        if MARKER in _norm(table.get_text(" ", strip=True)):
            return table
    return None


def retrieve_climate_table(s):
    """只接收 URL 字符串。有气候表返回 Tag；无表返回 None；网络/HTTP 错误抛异常。"""
    response = requests.get(s, headers=HEADERS, timeout=30)
    response.raise_for_status()  # 4xx/5xx 抛出 HTTPError
    # 传字节让 BeautifulSoup 自动识别编码（UTF-8），避免默认编码导致乱码
    return _find_climate_table(response.content)


def list_climate_table_row_names(s):
    """只接收 URL。返回行名列表（纯字符串）；无表返回 None。"""
    table = retrieve_climate_table(s)
    if table is None:
        return None

    names = []
    for tr in table.find_all("tr"):
        row_th = None
        for th in tr.find_all("th"):
            if th.get("scope") == "row":
                row_th = th
                break
        if row_th is None:
            # 无 scope="row" 时：该 tr 同时含数据单元格 <td>，则其首个 th 视为行名
            if tr.find("td") is not None and tr.find_all("th"):
                row_th = tr.find_all("th")[0]
            else:
                continue  # 月份表头、标题行、来源行等跳过
        name = _norm(row_th.get_text(" "))
        if not name:
            continue
        low = name.lower()
        if low.startswith("source") or low.startswith("climate data for"):
            continue
        names.append(name)
    return names

### 启动本地 HTTP 服务托管快照

快照文件（与本 notebook 一同提交）：

| 文件 | 还原的结构 | 用途 |
|---|---|---|
| `fixtures/madison.html` | `<caption>` + `scope="row"` 标准结构 | 主测试城市 Madison |
| `fixtures/los_angeles.html` | `<th colspan>` 标题、无 `scope`、`<td colspan>` 来源行 | 结构差异测试 |
| `fixtures/no_climate.html` | 普通表格、无气候表 | 验证返回 `None` |

In [10]:
import os
import functools
import http.server
import threading
import time

FIX_DIR = os.path.join(os.getcwd(), "fixtures")
assert os.path.isdir(FIX_DIR), "未找到 fixtures 目录，请在 hw04 目录下运行本 notebook"

PORT = 8765
handler = functools.partial(http.server.SimpleHTTPRequestHandler, directory=FIX_DIR)
server = http.server.ThreadingHTTPServer(("127.0.0.1", PORT), handler)
threading.Thread(target=server.serve_forever, daemon=True).start()
time.sleep(1)

BASE_URL = f"http://127.0.0.1:{PORT}"
print("本地服务已启动：", BASE_URL)

本地服务已启动： http://127.0.0.1:8765


### 测试 `retrieve_climate_table`：返回 Tag / None

In [11]:
# 两种结构都应返回 <table> Tag；无气候表页面返回 None
tag_madison = retrieve_climate_table(BASE_URL + "/madison.html")
tag_la = retrieve_climate_table(BASE_URL + "/los_angeles.html")
no_table = retrieve_climate_table(BASE_URL + "/no_climate.html")

print("Madison ->", tag_madison.name, tag_madison.get("class"))
print("Los Angeles(th标题结构) ->", tag_la.name)
print("无气候表页面 ->", no_table)

from bs4 import Tag
assert isinstance(tag_madison, Tag) and isinstance(tag_la, Tag)
assert tag_madison.name == "table" and tag_la.name == "table"
assert no_table is None
print("retrieve_climate_table 的 Tag / None 行为验证通过")

Madison -> table ['wikitable']
Los Angeles(th标题结构) -> table
无气候表页面 -> None
retrieve_climate_table 的 Tag / None 行为验证通过


127.0.0.1 - - [24/Nov/2026 16:54:54] "GET /madison.html HTTP/1.1" 200 -
127.0.0.1 - - [24/Nov/2026 16:54:54] "GET /los_angeles.html HTTP/1.1" 200 -
127.0.0.1 - - [24/Nov/2026 16:54:54] "GET /no_climate.html HTTP/1.1" 200 -


### 测试 `list_climate_table_row_names`：纯字符串行名

In [12]:
names_madison = list_climate_table_row_names(BASE_URL + "/madison.html")
names_la = list_climate_table_row_names(BASE_URL + "/los_angeles.html")
names_none = list_climate_table_row_names(BASE_URL + "/no_climate.html")

print("Madison 行名：")
for n in names_madison:
    print("  ", n)
print("Los Angeles 行名：")
for n in names_la:
    print("  ", n)
print("无气候表页面 ->", names_none)

# 每项必须是纯 Python 字符串，不能是 Tag 或含 HTML
assert all(isinstance(n, str) and not isinstance(n, Tag) for n in names_madison + names_la)
assert all("<" not in n for n in names_madison + names_la)
assert names_none is None
# 关键：标题行、月份、来源行均已被排除
assert all(not n.lower().startswith("source") for n in names_madison + names_la)
assert "Month" not in names_madison and "Jan" not in names_madison
print("行名均为纯字符串，且已正确排除标题/月份/来源行")

Madison 行名：
   Mean daily maximum °F (°C)
   Daily mean °F (°C)
   Mean daily minimum °F (°C)
   Average precipitation inches (mm)
   Average snowfall inches (cm)
   Average relative humidity (%)
   Mean monthly sunshine hours
Los Angeles 行名：
   Mean daily maximum °F (°C)
   Daily mean °F (°C)
   Mean daily minimum °F (°C)
   Average precipitation inches (mm)
无气候表页面 -> None
行名均为纯字符串，且已正确排除标题/月份/来源行


127.0.0.1 - - [24/Nov/2026 16:54:54] "GET /madison.html HTTP/1.1" 200 -
127.0.0.1 - - [24/Nov/2026 16:54:54] "GET /los_angeles.html HTTP/1.1" 200 -
127.0.0.1 - - [24/Nov/2026 16:54:54] "GET /no_climate.html HTTP/1.1" 200 -


### 异常处理验证：HTTP 错误与网络错误

- 访问本地服务上不存在的页面会得到 404，`raise_for_status()` 抛出 `HTTPError`；
- 访问当前不可达的真实维基 URL，连接/超时错误会被原样抛出（不被吞掉）。

In [13]:
# 1) HTTP 错误（404）应抛 HTTPError
try:
    retrieve_climate_table(BASE_URL + "/does_not_exist.html")
    raise AssertionError("应当抛出 HTTPError")
except requests.HTTPError as e:
    print("404 正确抛出 HTTPError，状态码：", e.response.status_code)

# 2) 真实维基 URL：当前网络不可达，应抛出 requests 连接/超时异常
host = "en." + "wikipedia.org"
real_url = "https://" + host + "/wiki/Madison,_Wisconsin"
print("尝试访问真实 URL：", real_url)
try:
    retrieve_climate_table(real_url)
    print("当前网络可访问，已成功取回真实气候表")
except requests.RequestException as e:
    print("网络错误按预期抛出：", type(e).__name__)

127.0.0.1 - - [24/Nov/2026 16:54:54] code 404, message File not found
127.0.0.1 - - [24/Nov/2026 16:54:54] "GET /does_not_exist.html HTTP/1.1" 404 -


404 正确抛出 HTTPError，状态码： 404
尝试访问真实 URL： https://en.wikipedia.org/wiki/Madison,_Wisconsin


网络错误按预期抛出： ConnectionError


### 实际使用的 URL 与访问日期

| 城市 | 目标 URL | 说明 |
|---|---|---|
| Madison, Wisconsin | `https://en.wikipedia.org/wiki/Madison,_Wisconsin` | 主测试（标准 caption 结构） |
| Los Angeles | `https://en.wikipedia.org/wiki/Los_Angeles` | 结构差异（th colspan，无 scope） |
| （无气候表对照） | 本地 `no_climate.html` 快照 | 验证返回 None |

- 访问日期：2026-11-24（Asia/Shanghai）
- 说明：因当前网络无法直连维基，函数逻辑通过同结构的本地快照经 HTTP 完整验证；真实 URL 的异常抛出行为也已实测。网络恢复后，将上述真实 URL 直接传入两个函数即可，无需改动代码。

In [14]:
# 关闭本地 HTTP 服务（daemon 线程，内核退出时也会自动结束）
server.shutdown()
print("本地 HTTP 服务已关闭")

本地 HTTP 服务已关闭


## 小结

1. 第一部分 12 个正则函数全部实现并通过断言，覆盖整串匹配、前导零清理、空格/下划线互换、日期转换、单词查找与内容删除等场景。
2. 第二部分两个函数严格只接收 URL：能区分气候表（返回 Tag）、无表（返回 None）与错误（抛异常），行名输出为干净的纯字符串并正确排除标题、月份与来源行。
3. 离线快照经本地 HTTP 服务完整验证了请求—解析—取行名的全流程，同时保留了真实 URL 的异常实测。